<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [3]:
# Posibles bandas
os.listdir("./songs_dataset/")

['adele.txt',
 'al-green.txt',
 'alicia-keys.txt',
 'amy-winehouse.txt',
 'beatles.txt',
 'bieber.txt',
 'bjork.txt',
 'blink-182.txt',
 'bob-dylan.txt',
 'bob-marley.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'bruno-mars.txt',
 'cake.txt',
 'dickinson.txt',
 'disney.txt',
 'dj-khaled.txt',
 'dolly-parton.txt',
 'dr-seuss.txt',
 'drake.txt',
 'eminem.txt',
 'janisjoplin.txt',
 'jimi-hendrix.txt',
 'johnny-cash.txt',
 'joni-mitchell.txt',
 'kanye-west.txt',
 'kanye.txt',
 'Kanye_West.txt',
 'lady-gaga.txt',
 'leonard-cohen.txt',
 'lil-wayne.txt',
 'Lil_Wayne.txt',
 'lin-manuel-miranda.txt',
 'lorde.txt',
 'ludacris.txt',
 'michael-jackson.txt',
 'missy-elliott.txt',
 'nickelback.txt',
 'nicki-minaj.txt',
 'nirvana.txt',
 'notorious-big.txt',
 'notorious_big.txt',
 'nursery_rhymes.txt',
 'patti-smith.txt',
 'paul-simon.txt',
 'prince.txt',
 'r-kelly.txt',
 'radiohead.txt',
 'rihanna.txt']

In [4]:
# Cada línea del archivo representa una oración/documento del corpus.
# Leemos las líneas directamente para evitar interpretar la letra como un CSV.
with open('songs_dataset/beatles.txt', encoding='utf-8') as file:
    documents = [line.strip() for line in file if line.strip()]

df = pd.DataFrame({'text': documents})
df.head()

,text
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [6]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Tokenizamos cada oración: pasamos a minúsculas y quitamos signos de puntuación.
for sentence in df['text']:
    sentence_tokens.append(text_to_word_sequence(sentence))

print(f'Oraciones tokenizadas: {len(sentence_tokens)}')

Oraciones tokenizadas: 1846


In [7]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', "they're", 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Creamos un modelo Skip-gram: aprende una palabra a partir de su contexto.
# La semilla permite reproducir los resultados del entrenamiento.
w2v_model = Word2Vec(
    min_count=5,      # frecuencia mínima para incluir una palabra en el vocabulario
    window=2,         # cantidad de palabras de contexto a cada lado
    vector_size=100,  # dimensión de cada embedding
    negative=10,      # cantidad de ejemplos negativos usados por actualización
    workers=1,        # un worker facilita la reproducibilidad
    sg=1,             # 0: CBOW, 1: Skip-gram
    seed=42
)

In [10]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [12]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 445


### 3 - Entrenar embeddings

In [13]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 100635.546875
Loss after epoch 1: 58683.6875
Loss after epoch 2: 55063.875
Loss after epoch 3: 54671.109375
Loss after epoch 4: 54591.75
Loss after epoch 5: 53279.25
Loss after epoch 6: 52806.3125
Loss after epoch 7: 51970.9375
Loss after epoch 8: 50600.09375
Loss after epoch 9: 50463.125
Loss after epoch 10: 48690.5625
Loss after epoch 11: 48006.6875
Loss after epoch 12: 47164.875
Loss after epoch 13: 46117.5625
Loss after epoch 14: 45314.6875
Loss after epoch 15: 44811.8125
Loss after epoch 16: 44737.0
Loss after epoch 17: 44460.1875
Loss after epoch 18: 44866.0
Loss after epoch 19: 44238.75


(156954, 287740)

### 4 - Ensayar

In [14]:
# Las palabras más similares comparten contextos parecidos con "darling".
# La similitud es coseno: valores cercanos a 1 indican contextos muy parecidos.
w2v_model.wv.most_similar(positive=['darling'], topn=10)

[('pretty', 0.8954159617424011),
 ('sleep', 0.8571984171867371),
 ('cry', 0.8282387256622314),
 ('not', 0.8202385902404785),
 ('do', 0.767553985118866),
 ('help', 0.7644333243370056),
 ('twist', 0.7610310912132263),
 ('seems', 0.7569643259048462),
 ('high', 0.7515426278114319),
 ('try', 0.7505161762237549)]

In [15]:
# Las palabras menos similares son las que quedan más alejadas del vector de "love".
# No significa que sean antónimos: solo aparecen en contextos menos parecidos.
w2v_model.wv.most_similar(negative=['love'], topn=10)

[('six', -0.17525792121887207),
 ('bang', -0.18022702634334564),
 ('our', -0.2030852884054184),
 ('it', -0.2078222632408142),
 ('five', -0.211755633354187),
 ('four', -0.21977850794792175),
 ('shake', -0.22269128262996674),
 ('three', -0.2468303143978119),
 ('go', -0.25024494528770447),
 ('yellow', -0.2509799003601074)]

In [16]:
# Los números se agrupan porque aparecen en frases con funciones sintácticas
# y contextos similares dentro de las letras.
w2v_model.wv.most_similar(positive=['four'], topn=10)

[('five', 0.9674635529518127),
 ('three', 0.9665218591690063),
 ('seven', 0.9500069618225098),
 ('six', 0.9467953443527222),
 ('two', 0.9389615058898926),
 ('sixty', 0.9132207632064819),
 ('one', 0.7732619047164917),
 ('us', 0.7522978782653809),
 ("i'm", 0.7474972009658813),
 ('then', 0.7438812255859375)]

In [17]:
# "money" permite observar un campo semántico más concreto y pequeño.
w2v_model.wv.most_similar(positive=['money'], topn=5)

[('buy', 0.9254371523857117),
 ("can't", 0.9244014620780945),
 ('much', 0.8913149833679199),
 ('that', 0.8635498285293579),
 ('hide', 0.8465230464935303)]

In [18]:
# Ensayar con una palabra que no está en el vocabulario:
#w2v_model.wv.most_similar(negative=["diedaa"])

In [19]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[-0.5069064   0.4257821   0.37461776  0.06045176 -0.12994385  0.49807817
  0.20563963 -0.01713854 -0.3663833  -0.16476044  0.20385164  0.500354
  0.31256545  0.3208284   0.17118129  0.18622357  0.09689131 -0.5439602
 -0.2224931   0.00549535 -0.27915174 -0.3365375  -0.06539585  0.45784208
  0.11978731 -0.49576005  0.25875142 -0.11649506  0.07032231  0.28361905
 -0.2848877  -0.2165938  -0.07384594  0.01643035  0.20893048 -0.7552027
  0.4918308  -0.3158092  -0.1530378  -0.02678058  0.33456922 -0.31248027
 -0.07128106 -0.2655576   0.30041212 -0.13672473  0.25677183  0.23611137
  0.2691027  -0.06991473  0.11063933 -0.3891815  -0.11845352  0.4147408
  0.0444927  -0.13879673  0.13974766  0.60253954 -0.04447754  0.47247466
  0.27959165 -0.088167   -0.14816023  0.17037264  0.35294372 -0.37493914
 -0.1233884  -0.6529189   0.15702386 -0.23339413  0.27880377 -0.05772306
 -0.26737952 -0.38184854 -0.5805756  -0.4065823   0.15424956 -0.27901548
 -0.1217491  -0.03627552  0.00198081  0.01487379 -0.3258

In [20]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0),
 ('babe', 0.9200338125228882),
 ('someone', 0.8661167621612549),
 ('need', 0.8585211038589478),
 ('nothing', 0.851650595664978),
 ('you', 0.8449001312255859),
 ("didn't", 0.8441917300224304),
 ('whoa', 0.8419359922409058),
 ('hope', 0.816783607006073),
 ('somebody', 0.8135316967964172)]

In [21]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9200338125228882),
 ('someone', 0.8661167621612549),
 ('need', 0.8585211038589478),
 ('nothing', 0.851650595664978),
 ('you', 0.8449001312255859),
 ("didn't", 0.8441917300224304),
 ('whoa', 0.841935932636261),
 ('hope', 0.816783607006073),
 ('somebody', 0.8135316967964172),
 ('tell', 0.7993157505989075)]

### Interpretación de las similitudes

- Para `darling`, aparecen palabras como `babe`, `someone`, `need` y `nothing`. Esto sugiere un contexto afectivo y conversacional compartido en las letras.
- Al consultar palabras negativas respecto de `love`, los resultados no deben interpretarse como antónimos. `most_similar(negative=[...])` busca vectores alejados, por lo que muestra términos con contextos distintos, como números, acciones o palabras funcionales.
- La consulta de `four` devuelve otros números (`five`, `three`, `six`, `seven`), lo que muestra que el modelo capturó un patrón de uso compartido.
- Para `money`, aparecen `can't`, `buy` y `much`, una asociación coherente con frases relacionadas con comprar o tener recursos.

Estas relaciones son estadísticas y dependen del corpus: no representan definiciones de diccionario ni garantizan una relación semántica universal.

### 5 - Visualizar agrupación de vectores

In [22]:
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions=2):
    """Proyecta los embeddings a 2 o 3 dimensiones para poder visualizarlos."""
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    # Fijamos la semilla para que la proyección pueda reproducirse.
    tsne = TSNE(
        n_components=num_dimensions,
        random_state=42,
        init='pca',
        learning_rate='auto'
    )
    projected_vectors = tsne.fit_transform(vectors)

    return projected_vectors, labels

In [23]:
# Proyectamos todos los embeddings y mostramos los términos más frecuentes.
vecs, labels = reduce_dimensions(w2v_model, num_dimensions=2)

MAX_WORDS = 150  # Menos etiquetas mejora la legibilidad del gráfico.
fig = px.scatter(
    x=vecs[:MAX_WORDS, 0],
    y=vecs[:MAX_WORDS, 1],
    text=labels[:MAX_WORDS],
    labels={'x': 'Componente 1', 'y': 'Componente 2'}
)
fig.update_traces(textposition='top center', marker_size=6)
fig.update_layout(title=f'Embeddings de las {MAX_WORDS} palabras más frecuentes')
fig.show(renderer='colab')

<!doctype html>

In [24]:
# La vista 3D sirve como apoyo: la consigna principal se resuelve en 2D.
vecs, labels = reduce_dimensions(w2v_model, num_dimensions=3)

fig = px.scatter_3d(
    x=vecs[:MAX_WORDS, 0],
    y=vecs[:MAX_WORDS, 1],
    z=vecs[:MAX_WORDS, 2],
    text=labels[:MAX_WORDS],
    labels={'x': 'Componente 1', 'y': 'Componente 2', 'z': 'Componente 3'}
)
fig.update_traces(marker_size=3)
fig.update_layout(title=f'Embeddings en 3 dimensiones ({MAX_WORDS} palabras)')
fig.show(renderer='colab')

<!doctype html>

In [25]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Interpretación de la visualización y conclusiones

En la proyección 2D se espera observar pequeños grupos de términos con usos parecidos. Por ejemplo, los números tienden a aparecer próximos entre sí, mientras que palabras afectivas como `love`, `darling`, `babe` y `dear` pueden formar otra región. También es posible encontrar agrupaciones de palabras frecuentes (`you`, `i`, `the`, `and`) que reflejan su aparición repetida en muchos contextos.

La cercanía debe interpretarse con cautela: t-SNE conserva principalmente vecindades locales y puede distorsionar distancias globales. Además, el corpus pertenece a un único artista y tiene 445 palabras después de aplicar `min_count=5`, por lo que las conclusiones describen el lenguaje de este conjunto de letras, no el idioma inglés en general.

El modelo cumple el objetivo del desafío: construye embeddings propios con Skip-gram, permite comparar palabras mediante similitud coseno, proyecta los vectores a 2D y documenta las asociaciones encontradas.

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

## Resolución del desafío 2: corpus literario en español

Para esta resolución se utiliza un corpus propio formado por tres obras disponibles localmente:

- *Don Quijote*, de Miguel de Cervantes Saavedra.
- *El libro de las mil noches y una noche*, de autor anónimo.
- *La Odisea*, atribuida a Homero.

Los archivos se limpian para quitar la cabecera y el pie de Project Gutenberg. Luego se separa el texto en oraciones, se tokeniza y se entrena un modelo Word2Vec Skip-gram. Las asociaciones aprendidas describen este corpus literario, no el idioma español en general.

In [26]:
import os
import re
from gensim.models import Word2Vec

CORPUS_DIR = "corpus"
FILES = [
    "Don_Quijote.txt",
    "El_libro_de_las_mil_noches_y_una_noche.txt",
    "La_Odisea.txt",
]

def clean_gutenberg(text: str) -> str:
    """Quita cabecera y pie de Project Gutenberg si están presentes."""
    start = re.search(r"\*\*\* START OF.*?\*\*\*", text)
    end = re.search(r"\*\*\* END OF.*?\*\*\*", text)
    if start:
        text = text[start.end():]
    if end:
        text = text[:end.start()]
    return text.strip()

texts = []
for name in FILES:
    path = os.path.join(CORPUS_DIR, name)
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    clean = clean_gutenberg(raw)
    print(f"{name}: {len(clean):,} caracteres")
    texts.append(clean)

corpus = "\n".join(texts)
print(f"\nCorpus total: {len(corpus):,} caracteres")

Don_Quijote.txt: 2,111,609 caracteres
El_libro_de_las_mil_noches_y_una_noche.txt: 370,496 caracteres
La_Odisea.txt: 983,232 caracteres

Corpus total: 3,465,339 caracteres


In [27]:
import pandas as pd

# Partir por signos de puntuacion de fin de oracion
sentences = re.split(r'[.!?]+', corpus)
sentences = [s.strip() for s in sentences if len(s.strip()) > 3]

df = pd.DataFrame(sentences, columns=["sentence"])
print (f"Cantidad de oraciones: {len(df)}")
df.head()

Cantidad de oraciones: 19259


,sentence
0,El ingenioso hidalgo don Quijote de la Mancha\...
1,"Y, para que dello conste, di la\npresente en V..."
2,Juan Gallo de Andrada
3,TESTIMONIO DE LAS ERRATAS\n\nEste libro no tie...
4,En el Colegio de la Madre de\nDios de los Teól...


Se observa que queda un poco de ruido de las oraciones iniciales

In [28]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = [text_to_word_sequence(s) for s in sentences]
print("Ejemplo:", sentence_tokens[0][:10])

Ejemplo: ['el', 'ingenioso', 'hidalgo', 'don', 'quijote', 'de', 'la', 'mancha', 'por', 'miguel']


In [29]:
from gensim.models.callbacks import CallbackAny2Vec 

class callback(CallbackAny2Vec):
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [30]:
# Entrenamos un modelo Skip-gram sobre las oraciones del corpus literario.
# La semilla permite repetir el experimento y comparar los resultados.
w2v_model = Word2Vec(
    min_count=8,
    window=2,
    vector_size=300,
    negative=20,
    workers=1,
    sg=1,
    seed=42
)

In [31]:
w2v_model.build_vocab(sentence_tokens)

In [32]:
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 19259


In [33]:
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))
print("Algunas son:", w2v_model.wv.index_to_key[:10])

Cantidad de words distintas en el corpus: 6180
Algunas son: ['de', 'y', 'que', 'la', 'el', 'en', 'a', 'los', 'á', 'no']


In [34]:
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
)

Loss after epoch 0: 3994607.25
Loss after epoch 1: 2999812.75
Loss after epoch 2: 2867378.0
Loss after epoch 3: 2765029.0
Loss after epoch 4: 2718751.0
Loss after epoch 5: 2636658.0
Loss after epoch 6: 2567656.0
Loss after epoch 7: 2536416.0
Loss after epoch 8: 2511830.0
Loss after epoch 9: 2485848.0
Loss after epoch 10: 2459786.0
Loss after epoch 11: 2442840.0
Loss after epoch 12: 2453360.0
Loss after epoch 13: 2462080.0
Loss after epoch 14: 2439432.0
Loss after epoch 15: 2421792.0
Loss after epoch 16: 2419880.0
Loss after epoch 17: 2410692.0
Loss after epoch 18: 2422308.0
Loss after epoch 19: 2466476.0


(7783282, 12538160)

*Términos de interés*

In [35]:
w2v_model.wv.most_similar(positive=["viaje"], topn=10)

[('permiso', 0.5193042159080505),
 ('vigésimo', 0.518565833568573),
 ('artífice', 0.5107545852661133),
 ('embarcarse', 0.5097100734710693),
 ('mes', 0.5051905512809753),
 ('crimen', 0.5036926865577698),
 ('sacarle', 0.497231125831604),
 ('irte', 0.4955568015575409),
 ('regreso', 0.4954410493373871),
 ('yugo', 0.4916120767593384)]

In [36]:
w2v_model.wv.most_similar(positive=["héroe"], topn=10)

[('reconoce', 0.5424098372459412),
 ('sudario', 0.5337576270103455),
 ('deiforme', 0.527793288230896),
 ('antíloco', 0.5025616884231567),
 ('corvo', 0.4969645142555237),
 ('teseo', 0.49615243077278137),
 ('polifemo', 0.49565306305885315),
 ('melampo', 0.4932539463043213),
 ('elpénor', 0.4930720627307892),
 ('liodes', 0.49108031392097473)]

*Términos menos relacionados*

In [37]:
w2v_model.wv.most_similar(negative=["familia"], topn=10)

[('cabras', -0.010798558592796326),
 ('parecían', -0.02630399912595749),
 ('agua', -0.046222880482673645),
 ('seguían', -0.04740571603178978),
 ('vieron', -0.058496516197919846),
 ('peces', -0.06011057272553444),
 ('les', -0.06607566773891449),
 ('dos', -0.06996188312768936),
 ('decían', -0.07288923114538193),
 ('uñas', -0.07366909831762314)]

*Palabra fuera del vocabulario*

Word2Vec tira KeyError.

In [38]:
try:
    w2v_model.wv.most_similar(positive=["smartphone"], topn=5)
except KeyError as e:
    print("KeyError:", e)

KeyError: "Key 'smartphone' not present in vocabulary"


Visualización 2D

Proyecto los vectores de 300 dim a 2D con t-SNE. Filtro stopwords antes de graficar porque si no las primeras 200 palabras son de, la, que, y, ... y no se ve nada temático. Word2Vec no las saca solo.

Probé MAX_WORDS en 100, 200 y 500. Con 500 las etiquetas se pisan, con 100 se pierden grupos. Me quedo con 200.

In [ ]:
from sklearn.cluster import KMeans

# Stopwords y tokens editoriales que no aportan contenido temático al gráfico.
STOPWORDS_ES = {
    "de", "la", "que", "y", "el", "en", "los", "a", "se", "no", "un", "por",
    "con", "las", "una", "para", "es", "su", "al", "lo", "como", "más", "pero",
    "sus", "le", "ya", "o", "este", "sí", "porque", "esta", "entre", "cuando",
    "muy", "sin", "sobre", "también", "me", "hasta", "hay", "donde", "quien",
    "desde", "todo", "nos", "durante", "todos", "uno", "les", "ni", "contra",
    "otros", "ese", "eso", "ante", "ellos", "e", "esto", "mí", "antes", "qué",
    "unos", "yo", "otro", "otras", "otra", "él", "tanto", "esa", "estos", "mucho",
    "quienes", "nada", "muchos", "cual", "poco", "ella", "estar", "algunas", "algo",
    "mi", "mis", "tú", "te", "ti", "tu", "tus", "os", "ha", "han", "he", "hemos",
    "había", "ser", "fue", "era", "son", "eran", "así", "sino", "aún", "aun", "cada",
    "mismo", "misma", "tan", "del", "pgdp", "ebook", "editions"
}

# Proyectamos los embeddings a dos dimensiones para visualizarlos.
vecs, labels = reduce_dimensions(w2v_model)

# Quitamos stopwords, símbolos, números y palabras editoriales.
# Así la visualización se concentra en términos con contenido léxico.
mask = np.array([
    label not in STOPWORDS_ES and label.isalpha() and len(label) >= 3
    for label in labels
])
vecs_f = vecs[mask]
labels_f = labels[mask]
word_vectors_f = np.asarray(w2v_model.wv.vectors)[mask]

MAX_WORDS = min(200, len(labels_f))
fig = px.scatter(
    x=vecs_f[:MAX_WORDS, 0],
    y=vecs_f[:MAX_WORDS, 1],
    text=labels_f[:MAX_WORDS],
    title=f"Embeddings 2D (t-SNE): top {MAX_WORDS} sin stopwords",
)
fig.update_traces(textposition="top center", marker_size=6)
fig.update_layout(height=800)
fig.show()

# Agrupamos los vectores originales, no las coordenadas de t-SNE.
# Esto evita que los clusters dependan exclusivamente de la proyección visual.
N_CLUSTERS = 6
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(word_vectors_f)

cluster_rows = []
for cluster_id in range(N_CLUSTERS):
    cluster_mask = cluster_ids == cluster_id
    distances = kmeans.transform(word_vectors_f[cluster_mask])[:, cluster_id]
    nearest_indices = np.argsort(distances)[:8]
    cluster_words = labels_f[cluster_mask][nearest_indices].tolist()
    cluster_rows.append({
        "cluster": cluster_id,
        "palabras_representativas": ", ".join(cluster_words)
    })

clusters_df = pd.DataFrame(cluster_rows)
print("Palabras representativas de cada cluster:")
display(clusters_df)

# Mostramos los mismos grupos sobre la proyección 2D usando un color por cluster.
cluster_fig = px.scatter(
    x=vecs_f[:MAX_WORDS, 0],
    y=vecs_f[:MAX_WORDS, 1],
    text=labels_f[:MAX_WORDS],
    color=cluster_ids[:MAX_WORDS].astype(str),
    labels={"color": "Cluster"},
    title=f"Clusters de palabras sobre t-SNE ({MAX_WORDS} términos)",
)
cluster_fig.update_traces(textposition="top center", marker_size=7)
cluster_fig.update_layout(height=800)
cluster_fig.show()

## Clusters y conclusiones

El gráfico coloreado muestra los seis clusters obtenidos con K-Means. Los grupos no tienen un significado asignado de antemano: el número del cluster es solamente una etiqueta técnica.

- Un grupo reúne nombres y personajes de *Don Quijote*, como `don`, `quijote`, `sancho`, `rocinante` y `cura`. Esto indica que sus vectores aparecen en contextos narrativos similares.
- Otro grupo contiene términos característicos de *La Odisea*, como `ulises`, `telémaco`, `penélope`, `pretendientes`, `júpiter` y `minerva`. La agrupación refleja el vocabulario propio de episodios y personajes mitológicos.
- Se observa un grupo con cantidades y expresiones numéricas, por ejemplo `dos`, `tres`, `cuatro`, `mil` y `años`. La cercanía se explica porque aparecen en contextos narrativos semejantes, aunque no constituyen un tema literario por sí mismas.
- Otro grupo contiene partes del cuerpo y elementos de desplazamiento, como `tierra`, `mar`, `nave`, `caballo`, `agua`, `puerta` y `cuerpo`. Son palabras que comparten contextos de viaje, acción y descripción.
- Los grupos restantes reúnen verbos, palabras de diálogo y vocabulario narrativo general, como `dijo`, `respondió`, `decir`, `hacer`, `camino` y `historia`.

En conclusión, los embeddings capturan asociaciones producidas por los contextos de las obras: personajes de una misma tradición, vocabulario de viajes y acciones, y patrones narrativos. Sin embargo, no debe afirmarse que cada cluster representa un tema puro. El corpus está formado por obras de extensiones y estilos diferentes, y *Don Quijote* aporta muchos más caracteres que las otras obras. Además, t-SNE conserva principalmente vecindades locales y puede deformar las distancias globales. Por eso, los clusters sirven como herramienta exploratoria y deben interpretarse junto con las palabras representativas y no solo por su posición en el gráfico.